# Mosaic Remote Inference — Colab experiment

This notebook is for an environment/account that explicitly permits running this inference service and using a Cloudflare Tunnel for it. It does **not** attempt to keep the Colab session alive or bypass runtime restrictions. The session is ephemeral; stop the experiment when done.

The Cloudflare side must already have a named Tunnel and hostname route configured to `http://localhost:8090`. Use a tunnel token for that tunnel. Do not use a Quick Tunnel for this SSE experiment.

The same high-entropy Mosaic bearer token must be set in this notebook runtime and in the local Mosaic backend. Do not save actual secrets in the notebook.

In [ ]:
# Install Ollama and the Cloudflare client in this ephemeral runtime.
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb

In [ ]:
import os, subprocess, time, urllib.request

# Start Ollama locally inside this runtime.
ollama_log = open('/tmp/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        print('Ollama is reachable on loopback.')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not become ready; inspect /tmp/ollama.log')

In [ ]:
# Download the configured model. Change this identifier if the available GPU/RAM supports another model.
MODEL = 'qwen2.5:3b'
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('Model prepared:', MODEL)

In [ ]:
# Fetch this repository and install its Python dependencies.
!git clone --depth 1 --branch feature/sse-streaming https://github.com/IGOR-SciDATA/mosaic-remote-inference.git /content/mosaic-remote-inference
%cd /content/mosaic-remote-inference
!pip install -r requirements.txt

In [ ]:
import getpass, secrets

# Paste the SAME value later into the local Mosaic backend environment.
# Input is hidden and is not written into notebook source/output by this cell.
REMOTE_TOKEN = getpass.getpass('MOSAIC_REMOTE_INFERENCE_TOKEN: ').strip()
if len(REMOTE_TOKEN) < 32:
    raise ValueError('Use a high-entropy token of at least 32 characters.')

os.environ['MOSAIC_REMOTE_INFERENCE_TOKEN'] = REMOTE_TOKEN
os.environ['MOSAIC_RUNTIME_BASE_URL'] = 'http://127.0.0.1:11434'
os.environ['MOSAIC_RUNTIME_MODEL'] = MODEL
os.environ['MOSAIC_INFERENCE_TIMEOUT_SECONDS'] = '300'
os.environ['MOSAIC_MAX_INPUT_CHARS'] = '24000'
os.environ['MOSAIC_MAX_MESSAGES'] = '40'
print('Service configuration loaded; token value was not printed.')

In [ ]:
# Start the FastAPI service on loopback only.
service_log = open('/tmp/mosaic-remote-inference.log', 'w')
service_proc = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '8090'],
    cwd='/content/mosaic-remote-inference',
    env=os.environ.copy(), stdout=service_log, stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8090/health/ready', timeout=3) as r:
            status = __import__('json').load(r)
        if status.get('status') == 'ready':
            print('Remote inference API is ready on loopback.')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Remote inference API did not become ready; inspect /tmp/mosaic-remote-inference.log')

## Connect the preconfigured Cloudflare Tunnel

In Cloudflare Zero Trust, create/configure a **named Tunnel** with a public hostname route to `http://localhost:8090`, then obtain its tunnel token. Do not put the tunnel token in notebook source or Git.

Only run the following cell if this serving setup is explicitly permitted for the current runtime/account. The tunnel token is read with hidden input. The hostname itself is configured in Cloudflare; this cell does not print a secret.

In [ ]:
CF_TUNNEL_TOKEN = getpass.getpass('Cloudflare named Tunnel token: ').strip()
if not CF_TUNNEL_TOKEN:
    raise ValueError('Tunnel token cannot be empty.')

tunnel_log = open('/tmp/cloudflared.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', CF_TUNNEL_TOKEN],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
print('Tunnel process started. Check its configured hostname in Cloudflare and test /health/ready.')

## Configure local Mosaic

1. Set `MOSAIC_REMOTE_INFERENCE_TOKEN` in the **local backend environment** to the same bearer secret used above.
2. In Mosaic Remote Inference Settings, enter the configured HTTPS hostname (not the localhost URL).
3. Create/enable a model with `provider=remote_inference` and `model_name` equal to `MOSAIC_RUNTIME_MODEL`.
4. Test `/health/ready`, then send a prompt and confirm the response appears token-by-token and the completed assistant message is present in local history.
5. Test a stopped session and verify Mosaic reports a failure rather than treating an interrupted response as complete.

The notebook does not persist Mosaic conversations or memory. When the runtime ends, the model process and service stop.

In [ ]:
# Optional shutdown cell. Run this when ending the experiment.
for proc in [tunnel_proc, service_proc, ollama_proc]:
    try:
        proc.terminate()
    except Exception:
        pass
print('Shutdown requested for tunnel, inference API, and Ollama.')